# PySpark functions reference

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-functions-reference.ipynb)

The functions from `pyspark.sql.functions` that most pipelines use, grouped like the
[official documentation](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html).
Each function, or each small family of related functions, has a card: what it does, its signature, what to use it
for, one thing to watch out for, and a runnable example with its output. 205 functions on 110 cards.

Use the [contents](#contents) table to jump to a group. Each group links to the lesson that explains when and why
to use its functions; this page is for looking them up.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 pandas pyarrow

## Setup and sample data

The first cell starts a local Spark session. The second creates the `customers` and `orders` tables shared across
the PySpark course, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data), plus a few
small tables used by several groups: `baskets` (arrays), `payloads` (JSON), `ads` (statistics) and the window
specification `by_customer`. Cards that need other data create it inline.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
import contextlib
import io
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

baskets = spark.createDataFrame(
    [(1, ["milk", "apple", "milk"], [1, 3, 2]), (2, ["bread"], [5]), (3, [], [])],
    "order_id int, items array<string>, qty array<int>",
)
payloads = spark.createDataFrame(
    [(1, '{"device": "ios", "geo": {"country": "DE"}, "tags": ["new", "promo"]}'),
     (2, '{"device": "web", "geo": {"country": "US"}}'),
     (3, "not json")],
    "event_id int, body string",
)
ads = spark.createDataFrame(
    [("north", 10, 120), ("north", 20, 190), ("north", 30, 310), ("north", 40, 390),
     ("south", 10, 200), ("south", 20, 180), ("south", 30, 220), ("south", None, 210)],
    "region string, spend int, sales int",
)
by_customer = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")


def plan_lines(df):
    # The lines df.explain() prints, so a card can pick out the operator it is about.
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain()
    return buffer.getvalue().splitlines()

## Contents

| group | functions | explained in |
| --- | --- | --- |
| [Column basics](#column-basics) | `col`, `column`, `lit`, `expr`, `broadcast` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#read-a-spark-column-expression) |
| [Conditional functions](#conditional-functions) | `when`, `otherwise`, `coalesce`, `nvl`, `ifnull`, `nvl2`, `nullif`, `nanvl` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#write-conditional-logic) |
| [Predicate functions](#predicate-functions) | `isnull`, `isnotnull`, `isnan`, `like`, `ilike`, `rlike`, `contains`, `startswith`, `endswith`, `isin`, `between` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#handle-nulls) |
| [Sort functions](#sort-functions) | `asc`, `desc`, `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#sort-with-nulls-first-or-last) |
| [Math functions](#math-functions) | `round`, `bround`, `ceil`, `floor`, `abs`, `sign`, `sqrt`, `pow`, `exp`, `log`, `log10`, `log2`, `greatest`, `least`, `rand`, `randn`, `pmod`, `try_add`, `try_subtract`, `try_multiply`, `try_divide` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#round-and-compute-numbers) |
| [String functions](#string-functions) | `upper`, `lower`, `initcap`, `trim`, `ltrim`, `rtrim`, `concat`, `concat_ws`, `length`, `substring`, `left`, `right`, `split`, `split_part`, `substring_index`, `regexp_extract`, `regexp_replace`, `lpad`, `rpad`, `format_string`, `format_number`, `translate`, `replace` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#work-with-strings) |
| [Date and timestamp functions](#date-and-timestamp-functions) | `current_date`, `current_timestamp`, `to_date`, `to_timestamp`, `date_format`, `year`, `quarter`, `month`, `dayofmonth`, `hour`, `minute`, `second`, `dayofweek`, `weekofyear`, `date_add`, `date_sub`, `add_months`, `datediff`, `months_between`, `date_trunc`, `trunc`, `last_day`, `next_day`, `unix_timestamp`, `from_unixtime`, `from_utc_timestamp`, `to_utc_timestamp`, `make_date`, `window` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#work-with-dates-and-timestamps) |
| [Hash and id functions](#hash-and-id-functions) | `md5`, `sha2`, `hash`, `xxhash64`, `monotonically_increasing_id`, `uuid` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#generate-ids-and-hashes) |
| [Array functions](#array-functions) | `array`, `size`, `array_contains`, `array_distinct`, `sort_array`, `array_join`, `flatten`, `arrays_zip`, `element_at`, `try_element_at`, `get`, `transform`, `filter`, `exists`, `forall`, `aggregate` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#use-small-arrays-and-maps-in-column-expressions) |
| [Struct and map functions](#struct-and-map-functions) | `struct`, `create_map`, `map_keys`, `map_values`, `map_contains_key` | [PySpark 8](pyspark-08-nested-and-semi-structured-data.ipynb) |
| [Generator functions](#generator-functions) | `explode`, `explode_outer`, `posexplode` | [PySpark 8](pyspark-08-nested-and-semi-structured-data.ipynb) |
| [JSON functions](#json-functions) | `schema_of_json`, `from_json`, `to_json`, `get_json_object`, `json_tuple` | [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#parse-and-build-json) |
| [Aggregate functions](#aggregate-functions) | `count`, `count_distinct`, `approx_count_distinct`, `sum`, `avg`, `mean`, `min`, `max`, `count_if`, `sum_distinct`, `product`, `try_sum`, `try_avg`, `collect_list`, `array_agg`, `collect_set`, `listagg`, `string_agg`, `first`, `last`, `any_value`, `max_by`, `min_by`, `mode`, `median`, `percentile`, `percentile_approx`, `approx_percentile`, `histogram_numeric`, `stddev`, `variance`, `skewness`, `kurtosis`, `corr`, `covar_samp`, `covar_pop`, `regr_slope`, `regr_intercept`, `regr_r2`, `regr_count`, `regr_avgx`, `regr_avgy`, `bool_and`, `every`, `bool_or`, `some`, `bit_or`, `bit_and`, `bit_xor`, `grouping`, `grouping_id` | [PySpark 7](pyspark-07-aggregations-and-windows.ipynb#aggregate-function-catalog) |
| [Window functions](#window-functions) | `row_number`, `rank`, `dense_rank`, `percent_rank`, `cume_dist`, `ntile`, `lag`, `lead`, `nth_value`, `first`, `last` | [PySpark window functions](pyspark-window-functions.ipynb) |
| [Misc and UDF functions](#misc-and-udf-functions) | `input_file_name`, `spark_partition_id`, `udf`, `pandas_udf` | [PySpark 9](pyspark-09-python-udfs-and-pandas.ipynb) |

## Column basics

Refer to columns, create constants and parse SQL snippets. Almost every other function takes the columns these
produce. Most functions also accept a plain column name as a string.

| function | what it does |
| --- | --- |
| `col`, `column` | Return a `Column` for a column name |
| `lit` | Turn a Python constant into a column, so it can sit next to real columns |
| `expr` | Parse a SQL expression into a column |
| `broadcast` | Mark a small DataFrame to be copied to every executor in a join, so the large side is not shuffled |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#read-a-spark-column-expression).

### `col`, `column`

Return a `Column` for a column name. `column` is an alias of `col`.

**Signature:** `F.col("name")` · `F.column("name")`
**Use it for:** Arithmetic, comparisons and methods such as `.cast`, `.alias`, `.isNull`.
**Watch out:** `"amount" * 2` is a Python error; write `F.col("amount") * 2`.

In [ ]:
orders.select(F.col("order_id"), (F.column("amount") * 2).alias("double_amount")).show(3)

### `lit`

Turn a Python constant into a column, so it can sit next to real columns.

**Signature:** `F.lit(value)`
**Use it for:** Constant columns (a currency, a load date) and literal arguments to other functions.
**Watch out:** `F.lit(1.2)` is a `double`; multiply a decimal by `F.lit(Decimal("1.2"))` to keep it exact.

In [ ]:
orders.select("order_id", F.lit("EUR").alias("currency"), F.lit(None).cast("string").alias("note")).show(3)

### `expr`

Parse a SQL expression into a column.

**Signature:** `F.expr("sql expression")`
**Use it for:** SQL-only syntax such as `try_cast`, `div` or `CASE WHEN`, or rules that read better in SQL.
**Watch out:** The string is parsed when the plan is built, so a typo fails at run time, not when you write it.

In [ ]:
orders.select(
    "order_id",
    F.expr("amount * 1.2").alias("gross"),
    F.expr("CASE WHEN amount >= 200 THEN 'large' ELSE 'small' END").alias("size"),
).show(3)

### `broadcast`

Mark a small DataFrame to be copied to every executor in a join, so the large side is not shuffled.

**Signature:** `F.broadcast(df)`
**Use it for:** Joining a large fact table to a small lookup table.
**Watch out:** Broadcast only tables that fit comfortably in executor memory.

In [ ]:
joined = orders.join(F.broadcast(customers), "customer_id")
print([line.strip()[:40] for line in plan_lines(joined) if "Broadcast" in line])

## Conditional functions

Choose a value per row: branch on conditions, or fall back when a value is missing.

| function | what it does |
| --- | --- |
| `when` / `otherwise` | SQL's `CASE WHEN`: the first true condition wins; `otherwise` gives the default |
| `coalesce` | Return the first non-null value from left to right |
| `nvl`, `ifnull` | Return the first argument, or the second when the first is null |
| `nvl2` | Return the second argument when the first is not null, otherwise the third |
| `nullif` | Return null when the two arguments are equal, otherwise the first |
| `nanvl` | Return the first argument, or the second when the first is `NaN` |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#write-conditional-logic).

### `when` / `otherwise`

SQL's `CASE WHEN`: the first true condition wins; `otherwise` gives the default.

**Signature:** `F.when(cond, value).when(cond, value).otherwise(default)`
**Use it for:** Tiers, flags and mapping source codes to clean values.
**Watch out:** Without `otherwise`, rows that match no branch get null.

In [ ]:
orders.select(
    "order_id", "amount",
    F.when(F.col("amount") >= 1000, "enterprise")
     .when(F.col("amount") >= 100, "mid_market")
     .otherwise("small").alias("tier"),
).show(4)

### `coalesce`

Return the first non-null value from left to right.

**Signature:** `F.coalesce(*cols)`
**Use it for:** Defaults for missing values and preferring one source over another.
**Watch out:** The arguments must have compatible types.

In [ ]:
contacts = spark.createDataFrame([(1, "555-01", None), (2, None, "555-02"), (3, None, None)], "id int, mobile string, home string")
contacts.select("id", F.coalesce("mobile", "home", F.lit("none")).alias("phone")).show()

### `nvl`, `ifnull`

Return the first argument, or the second when the first is null. Two-argument `coalesce`.

**Signature:** `F.nvl(col, default)` · `F.ifnull(col, default)`
**Use it for:** Code ported from SQL dialects that use these names.
**Watch out:** `default` must be a column: wrap constants in `F.lit`.

In [ ]:
customers.select("customer_id", "region", F.nvl("region", F.lit("unknown")).alias("nvl"),
                 F.ifnull("region", F.lit("unknown")).alias("ifnull")).where("customer_id IN (3, 4)").show()

### `nvl2`

Return the second argument when the first is not null, otherwise the third.

**Signature:** `F.nvl2(col, value_if_not_null, value_if_null)`
**Use it for:** Flags based on whether a value exists, ignoring the value itself.
**Watch out:** It tests only for null: an empty string counts as present.

In [ ]:
customers.select("customer_id", F.nvl2("region", F.lit("has region"), F.lit("missing")).alias("region_state")).where("customer_id IN (3, 4)").show()

### `nullif`

Return null when the two arguments are equal, otherwise the first.

**Signature:** `F.nullif(col, value)`
**Use it for:** Turning sentinels such as `''` or `-1` into real nulls.
**Watch out:** `nullif(trim(col), '')` also catches strings of spaces.

In [ ]:
raw = spark.createDataFrame([("a@x.com",), ("",), ("  ",)], "email string")
raw.select("email", F.nullif("email", F.lit("")).alias("nullif"), F.nullif(F.trim("email"), F.lit("")).alias("trimmed_nullif")).show()

### `nanvl`

Return the first argument, or the second when the first is `NaN`.

**Signature:** `F.nanvl(col, replacement)`
**Use it for:** Cleaning floating-point columns where `NaN` means invalid.
**Watch out:** `NaN` is not null: `coalesce` leaves it alone, and one `NaN` turns an `avg` into `NaN`.

In [ ]:
readings = spark.createDataFrame([(1.5,), (float("nan"),), (None,)], "value double")
readings.select("value", F.nanvl("value", F.lit(0.0)).alias("nanvl"), F.coalesce("value", F.lit(0.0)).alias("coalesce")).show()

## Predicate functions

Return true or false (or null) for each row. Use them in `where`, in `when`, or as boolean columns.

| function | what it does |
| --- | --- |
| `isnull`, `isnotnull` | Test whether a value is null |
| `isnan` | Test whether a floating-point value is `NaN` (not a number) |
| `like`, `ilike` | SQL wildcard match: `%` is any run of characters, `_` is one character |
| `rlike` | Java regular expression match, anywhere in the string unless anchored with `^` and `$` |
| `contains`, `startswith`, `endswith` | Plain substring tests, no wildcards or regex |
| `isin`, `between` | Membership in a list, and a closed range (both ends included) |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#handle-nulls).

### `isnull`, `isnotnull`

Test whether a value is null. Always true or false, never null.

**Signature:** `F.isnull(col)` · `F.isnotnull(col)` (or `col.isNull()`, `col.isNotNull()`)
**Use it for:** Finding missing values; `col == None` never works because comparisons with null are null.
**Watch out:** They do not detect `NaN`; use `isnan` for that.

In [ ]:
customers.select("customer_id", "region", F.isnull("region").alias("isnull"), F.isnotnull("region").alias("isnotnull")).where("customer_id IN (3, 4)").show()

### `isnan`

Test whether a floating-point value is `NaN` (not a number).

**Signature:** `F.isnan(col)`
**Use it for:** Data-quality checks on `double` columns.
**Watch out:** `isnan(null)` is false.

In [ ]:
readings.select("value", F.isnan("value").alias("isnan"), F.isnull("value").alias("isnull")).show()

### `like`, `ilike`

SQL wildcard match: `%` is any run of characters, `_` is one character. `ilike` ignores case.

**Signature:** `F.col(c).like("pattern")` · `F.col(c).ilike("pattern")`
**Use it for:** Simple prefix, suffix and contains patterns that analysts already write in SQL.
**Watch out:** The pattern must match the whole string, so `%` is needed at the ends for a contains test.

In [ ]:
files = spark.createDataFrame([("SKU-042",), ("sku-7",), ("red.csv",)], "label string")
files.select("label", F.col("label").like("SKU-%").alias("like"), F.col("label").ilike("sku-%").alias("ilike")).show()

### `rlike`

Java regular expression match, anywhere in the string unless anchored with `^` and `$`.

**Signature:** `F.col(c).rlike("regex")`
**Use it for:** Validating formats such as codes, emails and phone numbers.
**Watch out:** Escape regex characters: a literal dot is `\.`.

In [ ]:
files.select("label", F.col("label").rlike(r"^SKU-[0-9]+$").alias("valid_sku"), F.col("label").rlike(r"\.csv$").alias("is_csv")).show()

### `contains`, `startswith`, `endswith`

Plain substring tests, no wildcards or regex.

**Signature:** `F.col(c).contains("x")` · `.startswith("x")` · `.endswith("x")`
**Use it for:** The common case; faster and clearer than a regex.
**Watch out:** They are case-sensitive; compare `F.lower(col)` for a case-insensitive test.

In [ ]:
files.select("label", F.col("label").contains("-").alias("contains"), F.col("label").startswith("SKU").alias("startswith"),
             F.col("label").endswith(".csv").alias("endswith")).show()

### `isin`, `between`

Membership in a list, and a closed range (both ends included). Column methods rather than functions.

**Signature:** `F.col(c).isin("a", "b")` · `F.col(c).between(low, high)`
**Use it for:** Status filters and value ranges.
**Watch out:** `isin` with a null value returns null, not false.

In [ ]:
orders.select("order_id", "status", "amount", F.col("status").isin("paid", "shipped").alias("settled"),
              F.col("amount").between(80, 150).alias("mid_range")).show(4)

## Sort functions

Set the direction of `orderBy` and of window ordering, and choose where nulls go. By default ascending order puts
nulls first and descending order puts them last.

| function | what it does |
| --- | --- |
| `asc`, `desc` | Ascending or descending order |
| `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last` | The same directions with an explicit position for nulls |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#sort-with-nulls-first-or-last).

### `asc`, `desc`

Ascending or descending order.

**Signature:** `F.asc("col")` · `F.desc("col")` (or `F.col(c).asc()`, `.desc()`)
**Use it for:** `orderBy` and `Window.orderBy`.
**Watch out:** Add a unique tie-breaker column when the order must be the same on every run.

In [ ]:
print("asc: ", [r.region for r in customers.orderBy(F.asc("region"), "customer_id").collect()])
print("desc:", [r.region for r in customers.orderBy(F.desc("region"), "customer_id").collect()])

### `asc_nulls_first`, `asc_nulls_last`, `desc_nulls_first`, `desc_nulls_last`

The same directions with an explicit position for nulls.

**Signature:** `F.asc_nulls_last("col")` and the other three
**Use it for:** Reports where unknown values should not lead, and rankings where nulls should rank last.
**Watch out:** Other engines have different defaults (PostgreSQL puts nulls last in ascending order).

In [ ]:
print("asc_nulls_last:  ", [r.region for r in customers.orderBy(F.asc_nulls_last("region"), "customer_id").collect()])
print("desc_nulls_first:", [r.region for r in customers.orderBy(F.desc_nulls_first("region"), "customer_id").collect()])

## Math functions

Round, compute and compare numbers. Spark 4 runs in ANSI mode: overflow and division by zero raise errors, and the
`try_` functions return null instead.

| function | what it does |
| --- | --- |
| `round`, `bround` | Round to `scale` decimals |
| `ceil`, `floor` | Round up or down to a whole number |
| `abs`, `sign` | Absolute value, and the sign as -1.0, 0.0 or 1.0 |
| `sqrt`, `pow`, `exp`, `log`, `log10`, `log2` | Square root, power, e to the power, and logarithms (`log(x)` is natural; `log(base, x)` takes the base first) |
| `greatest`, `least` | The largest or smallest value **across columns** in the same row, skipping nulls |
| `rand`, `randn` | Random numbers per row: uniform in [0, 1), or standard normal |
| `pmod` | Positive remainder: like `%`, but never negative for a positive divisor |
| `try_add`, `try_subtract`, `try_multiply`, `try_divide` | Arithmetic that returns null on overflow or division by zero instead of failing the job |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#round-and-compute-numbers).

### `round`, `bround`

Round to `scale` decimals. `round` rounds halves up (away from zero); `bround` rounds halves to the even neighbour.

**Signature:** `F.round(col, scale=0)` · `F.bround(col, scale=0)`
**Use it for:** Display values and reporting rules that name a rounding mode.
**Watch out:** Round at the end of a calculation, not at every step: rounding before summing changes totals.

In [ ]:
halves = spark.createDataFrame([(D("0.5"),), (D("1.5"),), (D("2.5"),), (D("2.345"),)], "value decimal(5,3)")
halves.select("value", F.round("value").alias("round"), F.bround("value").alias("bround"), F.round("value", 2).alias("round_2")).show()

### `ceil`, `floor`

Round up or down to a whole number.

**Signature:** `F.ceil(col)` · `F.floor(col)`
**Use it for:** Bucket boundaries and "how many boxes do I need" calculations.
**Watch out:** For negative numbers `floor(-1.5)` is -2, not -1.

In [ ]:
spark.createDataFrame([(1.2,), (-1.5,)], "x double").select("x", F.ceil("x").alias("ceil"), F.floor("x").alias("floor")).show()

### `abs`, `sign`

Absolute value, and the sign as -1.0, 0.0 or 1.0.

**Signature:** `F.abs(col)` · `F.sign(col)`
**Use it for:** Distances and the direction of a change.
**Watch out:** `abs` of the smallest `int` overflows in ANSI mode.

In [ ]:
spark.createDataFrame([(-4,), (0,), (7,)], "delta int").select("delta", F.abs("delta").alias("abs"), F.sign("delta").alias("sign")).show()

### `sqrt`, `pow`, `exp`, `log`, `log10`, `log2`

Square root, power, e to the power, and logarithms (`log(x)` is natural; `log(base, x)` takes the base first).

**Signature:** `F.sqrt(col)` · `F.pow(col, p)` · `F.exp(col)` · `F.log(col)` · `F.log(base, col)` · `F.log10(col)` · `F.log2(col)`
**Use it for:** Growth rates, scaling skewed values, distance formulas.
**Watch out:** The log of zero or a negative number is null.

In [ ]:
nums = spark.createDataFrame([(100.0,), (8.0,), (0.0,)], "x double")
nums.select("x", F.sqrt("x").alias("sqrt"), F.pow("x", 2).alias("pow_2"), F.round(F.log("x"), 3).alias("ln"),
            F.log10("x").alias("log10"), F.log2("x").alias("log2"), F.round(F.exp(F.lit(1)), 4).alias("e")).show()

### `greatest`, `least`

The largest or smallest value **across columns** in the same row, skipping nulls.

**Signature:** `F.greatest(*cols)` · `F.least(*cols)`
**Use it for:** Clamping values to a range and picking the later of two dates.
**Watch out:** Not the same as `max` and `min`, which compare **across rows**.

In [ ]:
bounds = spark.createDataFrame([(-3, 5), (12, None)], "x int, y int")
bounds.select("x", "y", F.greatest("x", "y").alias("greatest"), F.least("x", "y").alias("least"),
              F.least(F.greatest("x", F.lit(0)), F.lit(10)).alias("x_clamped_0_10")).show()

### `rand`, `randn`

Random numbers per row: uniform in [0, 1), or standard normal.

**Signature:** `F.rand(seed=None)` · `F.randn(seed=None)`
**Use it for:** Sampling and test/control splits.
**Watch out:** Results repeat only with the same seed **and** the same partitioning.

In [ ]:
spark.range(3, numPartitions=1).select("id", F.round(F.rand(seed=7), 3).alias("rand"), F.round(F.randn(seed=7), 3).alias("randn")).show()

### `pmod`

Positive remainder: like `%`, but never negative for a positive divisor.

**Signature:** `F.pmod(dividend, divisor)`
**Use it for:** Bucket numbers and weekday rotation from possibly negative numbers.
**Watch out:** `%` keeps the sign of the dividend: `-3 % 7` is -3, `pmod(-3, 7)` is 4.

In [ ]:
spark.createDataFrame([(-3,), (10,)], "x int").select("x", (F.col("x") % 7).alias("x % 7"), F.pmod("x", F.lit(7)).alias("pmod")).show()

### `try_add`, `try_subtract`, `try_multiply`, `try_divide`

Arithmetic that returns null on overflow or division by zero instead of failing the job.

**Signature:** `F.try_add(a, b)` · `F.try_subtract(a, b)` · `F.try_multiply(a, b)` · `F.try_divide(a, b)`
**Use it for:** Ratios with a possible zero denominator and data with occasional extreme values.
**Watch out:** Count the nulls these produce; a silent null is still a data problem.

In [ ]:
limits = spark.createDataFrame([(2147483647, 2, 0)], "big int, two int, zero int")
limits.select(F.try_add("big", "two").alias("try_add"), F.try_multiply("big", "two").alias("try_multiply"),
              F.try_divide("two", "zero").alias("try_divide"), F.try_subtract(F.lit(10), "two").alias("try_subtract")).show()

## String functions

Clean, combine, slice and parse text. They return null for a null input, except `concat_ws`, which skips nulls.
Positions are 1-based.

| function | what it does |
| --- | --- |
| `upper`, `lower`, `initcap` | Change case: all upper, all lower, or the first letter of each word upper |
| `trim`, `ltrim`, `rtrim` | Remove spaces from both ends, the left end or the right end |
| `concat`, `concat_ws` | Join strings |
| `length` | Number of characters in a string (or bytes in a binary value) |
| `substring`, `left`, `right` | Take part of a string by position: `substring(s, pos, len)`, or the first or last `n` characters |
| `split`, `split_part`, `substring_index` | Split on a delimiter: `split` returns an array (the delimiter is a regex); `split_part` returns the nth piece; `substring_index` returns everything before (or after) the nth delimiter |
| `regexp_extract` | Return the text matched by one group of a regular expression |
| `regexp_replace` | Replace every match of a regular expression |
| `lpad`, `rpad` | Pad a string on the left or right to a fixed width |
| `format_string`, `format_number` | printf-style formatting, and numbers as text with thousands separators |
| `translate`, `replace` | `translate` maps characters one to one (or deletes them); `replace` swaps whole substrings |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#work-with-strings).

### `upper`, `lower`, `initcap`

Change case: all upper, all lower, or the first letter of each word upper.

**Signature:** `F.upper(col)` · `F.lower(col)` · `F.initcap(col)`
**Use it for:** Case-insensitive join keys (`lower(trim(email))`) and display names.
**Watch out:** `initcap` lowercases the rest of each word, so `McDonald` becomes `Mcdonald`.

In [ ]:
names = spark.createDataFrame([("ada LOVELACE",), ("McDonald",), (None,)], "name string")
names.select("name", F.upper("name").alias("upper"), F.lower("name").alias("lower"), F.initcap("name").alias("initcap")).show()

### `trim`, `ltrim`, `rtrim`

Remove spaces from both ends, the left end or the right end.

**Signature:** `F.trim(col)` · `F.ltrim(col)` · `F.rtrim(col)`
**Use it for:** Cleaning keys and codes typed by people.
**Watch out:** Spaces inside the string stay; use `regexp_replace` to collapse them.

In [ ]:
padded = spark.createDataFrame([("  ab  c  ",)], "s string")
padded.select(F.concat(F.lit("["), F.trim("s"), F.lit("]")).alias("trim"), F.concat(F.lit("["), F.ltrim("s"), F.lit("]")).alias("ltrim"),
              F.concat(F.lit("["), F.rtrim("s"), F.lit("]")).alias("rtrim")).show()

### `concat`, `concat_ws`

Join strings. `concat_ws` puts a separator between them and skips nulls.

**Signature:** `F.concat(*cols)` · `F.concat_ws(sep, *cols)`
**Use it for:** Display names and composite keys before hashing.
**Watch out:** One null input makes `concat` return null for the whole row.

In [ ]:
people = spark.createDataFrame([("Grace", "Hopper"), ("Ada", None)], "first string, last string")
people.select(F.concat("first", F.lit(" "), "last").alias("concat"), F.concat_ws(" ", "first", "last").alias("concat_ws")).show()

### `length`

Number of characters in a string (or bytes in a binary value).

**Signature:** `F.length(col)`
**Use it for:** Validating fixed-width codes and finding truncated values.
**Watch out:** Trailing spaces count; trim first if they do not matter.

In [ ]:
spark.createDataFrame([("SKU-042",), ("SKU-42 ",)], "code string").select("code", F.length("code").alias("length")).show()

### `substring`, `left`, `right`

Take part of a string by position: `substring(s, pos, len)`, or the first or last `n` characters.

**Signature:** `F.substring(col, pos, len)` · `F.left(col, n)` · `F.right(col, n)`
**Use it for:** Fixed-position codes such as a country prefix.
**Watch out:** Positions start at 1; `left` and `right` take the length as a column, so pass `F.lit(n)`.

In [ ]:
codes = spark.createDataFrame([("UK-LON-0042",)], "code string")
codes.select(F.substring("code", 4, 3).alias("substring_4_3"), F.left("code", F.lit(2)).alias("left_2"), F.right("code", F.lit(4)).alias("right_4")).show()

### `split`, `split_part`, `substring_index`

Split on a delimiter: `split` returns an array (the delimiter is a regex); `split_part` returns the nth piece; `substring_index` returns everything before (or after) the nth delimiter.

**Signature:** `F.split(col, pattern)` · `F.split_part(col, F.lit(delim), F.lit(n))` · `F.substring_index(col, delim, count)`
**Use it for:** Parsing paths, codes and delimited fields.
**Watch out:** `split` treats the delimiter as a regex, so a literal pipe is `r"\|"`; `split_part` does not.

In [ ]:
paths = spark.createDataFrame([("Kitchen|Knives|Chef",)], "path string")
paths.select(F.split("path", r"\|").alias("split"), F.split_part("path", F.lit("|"), F.lit(-1)).alias("last_part"),
             F.substring_index("path", "|", 2).alias("first_two")).show(truncate=False)

### `regexp_extract`

Return the text matched by one group of a regular expression.

**Signature:** `F.regexp_extract(col, pattern, group)`
**Use it for:** Pulling an id, a city or a number out of free text.
**Watch out:** No match gives an empty string, not null; wrap it in `nullif(..., '')`.

In [ ]:
codes2 = spark.createDataFrame([("UK-LON-0042",), ("unassigned",)], "code string")
codes2.select("code", F.regexp_extract("code", r"^([A-Z]{2})-([A-Z]{3})-(\d+)$", 2).alias("city"),
              F.regexp_extract("code", r"(\d+)$", 1).alias("number")).show()

### `regexp_replace`

Replace every match of a regular expression.

**Signature:** `F.regexp_replace(col, pattern, replacement)`
**Use it for:** Masking digits, collapsing whitespace, removing punctuation.
**Watch out:** Use `$1` in the replacement to reuse a captured group.

In [ ]:
messy = spark.createDataFrame([("call  555 0100   now",)], "s string")
messy.select(F.regexp_replace("s", r"\s+", " ").alias("collapse_spaces"), F.regexp_replace("s", r"\d", "#").alias("mask_digits")).show(truncate=False)

### `lpad`, `rpad`

Pad a string on the left or right to a fixed width.

**Signature:** `F.lpad(col, len, pad)` · `F.rpad(col, len, pad)`
**Use it for:** Zero-padded ids and fixed-width exports.
**Watch out:** Both **truncate** strings longer than `len`; check lengths first if that would corrupt a key.

In [ ]:
ids = spark.createDataFrame([("7",), ("123456",)], "id string")
ids.select("id", F.lpad("id", 5, "0").alias("lpad_5"), F.rpad("id", 5, ".").alias("rpad_5")).show()

### `format_string`, `format_number`

printf-style formatting, and numbers as text with thousands separators.

**Signature:** `F.format_string("fmt", *cols)` · `F.format_number(col, decimals)`
**Use it for:** Labels and display values in exports.
**Watch out:** Both return strings; format only at the end, for display.

In [ ]:
orders.select("order_id", F.format_string("ORD-%05d", "order_id").alias("label"), F.format_number(F.col("amount") * 1000, 2).alias("amount_x1000")).show(3)

### `translate`, `replace`

`translate` maps characters one to one (or deletes them); `replace` swaps whole substrings. Neither uses regex.

**Signature:** `F.translate(col, "from_chars", "to_chars")` · `F.replace(col, F.lit(search), F.lit(replacement))`
**Use it for:** Stripping punctuation from phone numbers; renaming tokens.
**Watch out:** `translate` deletes characters that have no counterpart in `to_chars`.

In [ ]:
phones = spark.createDataFrame([("(512) 555-0100",)], "phone string")
phones.select(F.translate("phone", "()- ", "").alias("translate"), F.replace("phone", F.lit("555"), F.lit("XXX")).alias("replace")).show()

## Date and timestamp functions

Parse, format, take apart and shift dates and timestamps. A `timestamp` is an instant shown in the session time
zone (UTC in this notebook); a `date` is a calendar day. Format letters are case-sensitive: `MM` is month, `mm`
is minute.

| function | what it does |
| --- | --- |
| `current_date`, `current_timestamp` | The date and instant when the query started, the same on every row of one query |
| `to_date`, `to_timestamp` | Parse text into a date or timestamp, with an optional pattern |
| `date_format` | Format a date or timestamp as text with a pattern |
| `year`, `quarter`, `month`, `dayofmonth` | Extract calendar parts as integers |
| `hour`, `minute`, `second` | Extract the time of day from a timestamp |
| `dayofweek`, `weekofyear` | Day of the week (Sunday = 1 to Saturday = 7) and ISO week number (weeks start on Monday) |
| `date_add`, `date_sub`, `add_months` | Move a date by days or months |
| `datediff`, `months_between` | Whole days between two dates (`end` first), and fractional months between two dates |
| `date_trunc`, `trunc` | Truncate to the start of a unit |
| `last_day`, `next_day` | The last day of the date's month, and the next given weekday strictly after the date |
| `unix_timestamp`, `from_unixtime` | Seconds since 1970-01-01 UTC, and back to formatted text |
| `from_utc_timestamp`, `to_utc_timestamp` | Convert between UTC and a named time zone's wall-clock time |
| `make_date` | Build a date from year, month and day columns |
| `window` | Assign each row to a fixed (or sliding) time bucket; returns a struct with `start` and `end` |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#work-with-dates-and-timestamps).

### `current_date`, `current_timestamp`

The date and instant when the query started, the same on every row of one query.

**Signature:** `F.current_date()` · `F.current_timestamp()`
**Use it for:** Audit columns such as `loaded_at`.
**Watch out:** Results change on every run; pass the business date into a job instead of calling `current_date()`.

In [ ]:
spark.range(2).select("id", F.current_date().alias("today"), F.current_timestamp().alias("now")).printSchema()

### `to_date`, `to_timestamp`

Parse text into a date or timestamp, with an optional pattern.

**Signature:** `F.to_date(col, "pattern")` · `F.to_timestamp(col, "pattern")`
**Use it for:** Typing raw dates such as `03/01/2026` or `2026-01-31 18:45`.
**Watch out:** An input that does not match raises an error in ANSI mode; `F.try_to_timestamp` returns null.

In [ ]:
raw_dates = spark.createDataFrame([("03/01/2026", "2026-01-31 18:45")], "d string, ts string")
raw_dates.select(F.to_date("d", "dd/MM/yyyy").alias("to_date"), F.to_timestamp("ts", "yyyy-MM-dd HH:mm").alias("to_timestamp")).show()

### `date_format`

Format a date or timestamp as text with a pattern.

**Signature:** `F.date_format(col, "pattern")`
**Use it for:** Labels such as `2026-01` or `Monday`.
**Watch out:** The result is a string, so it no longer sorts or compares as a date.

In [ ]:
orders.select("order_ts", F.date_format("order_ts", "yyyy-MM").alias("month"), F.date_format("order_ts", "EEEE").alias("weekday"),
              F.date_format("order_ts", "dd MMM HH:mm").alias("short")).show(3)

### `year`, `quarter`, `month`, `dayofmonth`

Extract calendar parts as integers.

**Signature:** `F.year(col)` · `F.quarter(col)` · `F.month(col)` · `F.dayofmonth(col)`
**Use it for:** Grouping and partition columns.
**Watch out:** For grouping by month, `date_trunc` keeps one sortable value instead of two columns.

In [ ]:
orders.select("order_ts", F.year("order_ts").alias("year"), F.quarter("order_ts").alias("quarter"),
              F.month("order_ts").alias("month"), F.dayofmonth("order_ts").alias("day")).show(3)

### `hour`, `minute`, `second`

Extract the time of day from a timestamp.

**Signature:** `F.hour(col)` · `F.minute(col)` · `F.second(col)`
**Use it for:** Hour-of-day traffic patterns.
**Watch out:** They read the time in the session time zone.

In [ ]:
orders.select("order_ts", F.hour("order_ts").alias("hour"), F.minute("order_ts").alias("minute"), F.second("order_ts").alias("second")).show(3)

### `dayofweek`, `weekofyear`

Day of the week (Sunday = 1 to Saturday = 7) and ISO week number (weeks start on Monday).

**Signature:** `F.dayofweek(col)` · `F.weekofyear(col)`
**Use it for:** Weekday reports and weekly grouping.
**Watch out:** Near New Year the ISO week can belong to the previous or next year; group by `date_trunc("week", ts)` instead.

In [ ]:
orders.select("order_ts", F.date_format("order_ts", "E").alias("day"), F.dayofweek("order_ts").alias("dayofweek"),
              F.weekofyear("order_ts").alias("iso_week")).show(3)

### `date_add`, `date_sub`, `add_months`

Move a date by days or months.

**Signature:** `F.date_add(col, days)` · `F.date_sub(col, days)` · `F.add_months(col, months)`
**Use it for:** Due dates, look-back windows and month offsets.
**Watch out:** Adding a month to 31 January lands on the last day of February.

In [ ]:
jan31 = spark.createDataFrame([(date(2026, 1, 31),)], "d date")
jan31.select("d", F.date_add("d", 1).alias("date_add"), F.date_sub("d", 31).alias("date_sub"), F.add_months("d", 1).alias("add_months")).show()

### `datediff`, `months_between`

Whole days between two dates (`end` first), and fractional months between two dates.

**Signature:** `F.datediff(end, start)` · `F.months_between(end, start)`
**Use it for:** Days to ship, customer age in months.
**Watch out:** Swapping the arguments flips the sign.

In [ ]:
spans = spark.createDataFrame([(date(2026, 1, 5), date(2026, 3, 20))], "start date, end date")
spans.select(F.datediff("end", "start").alias("datediff"), F.datediff("start", "end").alias("swapped"),
             F.round(F.months_between("end", "start"), 2).alias("months_between")).show()

### `date_trunc`, `trunc`

Truncate to the start of a unit. `date_trunc(unit, ts)` returns a timestamp; `trunc(date, unit)` returns a date. Note the argument order.

**Signature:** `F.date_trunc("month", col)` · `F.trunc(col, "month")`
**Use it for:** Grouping by week, month or quarter with one sortable key.
**Watch out:** `date_trunc` takes the unit first and `trunc` takes it second; units include `"year"`, `"quarter"`, `"month"`, `"week"`, `"day"`, `"hour"`.

In [ ]:
orders.select("order_ts", F.date_trunc("week", "order_ts").alias("week_start"), F.date_trunc("hour", "order_ts").alias("hour_start"),
              F.trunc(F.to_date("order_ts"), "month").alias("month_start")).show(3)

### `last_day`, `next_day`

The last day of the date's month, and the next given weekday strictly after the date.

**Signature:** `F.last_day(col)` · `F.next_day(col, "Mon")`
**Use it for:** Month-end reporting and "next Monday" schedules.
**Watch out:** `next_day` never returns the same day: for a Monday it returns the following Monday.

In [ ]:
orders.select(F.to_date("order_ts").alias("day"), F.last_day("order_ts").alias("last_day"), F.next_day(F.to_date("order_ts"), "Mon").alias("next_monday")).show(3)

### `unix_timestamp`, `from_unixtime`

Seconds since 1970-01-01 UTC, and back to formatted text.

**Signature:** `F.unix_timestamp(col)` · `F.from_unixtime(seconds, "pattern")`
**Use it for:** Systems that exchange epoch seconds.
**Watch out:** For epoch milliseconds use `timestamp_millis` and `unix_millis`.

In [ ]:
orders.select("order_ts", F.unix_timestamp("order_ts").alias("epoch_s"),
              F.from_unixtime(F.unix_timestamp("order_ts"), "yyyy-MM-dd HH:mm").alias("from_unixtime")).show(3)

### `from_utc_timestamp`, `to_utc_timestamp`

Convert between UTC and a named time zone's wall-clock time.

**Signature:** `F.from_utc_timestamp(col, "America/Chicago")` · `F.to_utc_timestamp(col, "America/Chicago")`
**Use it for:** Business dates in a local time zone.
**Watch out:** Use named zones, not fixed offsets, so daylight saving time is handled.

In [ ]:
orders.where("order_id = 12").select("order_ts", F.from_utc_timestamp("order_ts", "America/Chicago").alias("chicago_time"),
    F.to_utc_timestamp(F.from_utc_timestamp("order_ts", "America/Chicago"), "America/Chicago").alias("back_to_utc")).show()

### `make_date`

Build a date from year, month and day columns.

**Signature:** `F.make_date(year, month, day)`
**Use it for:** Sources that store dates as separate integers.
**Watch out:** An impossible date such as 30 February raises an error in ANSI mode.

In [ ]:
spark.createDataFrame([(2026, 1, 31)], "y int, m int, d int").select(F.make_date("y", "m", "d").alias("make_date")).show()

### `window`

Assign each row to a fixed (or sliding) time bucket; returns a struct with `start` and `end`.

**Signature:** `F.window(ts, "1 day")` · `F.window(ts, "1 hour", "30 minutes")`
**Use it for:** Counts per hour or per day, in batch and streaming.
**Watch out:** Only buckets with rows appear; this grouping `window` is unrelated to `Window` for window functions.

In [ ]:
orders.groupBy(F.window("order_ts", "2 days")).agg(F.count("*").alias("orders")).select("window.start", "window.end", "orders").orderBy("start").show(3)

## Hash and id functions

Generate deterministic hashes for keys and change detection, or ids that are unique only within one run.

| function | what it does |
| --- | --- |
| `md5`, `sha2` | Cryptographic hashes as hex strings: MD5 (128-bit) and SHA-2 with 224, 256, 384 or 512 bits |
| `hash`, `xxhash64` | Fast non-cryptographic hashes of one or more columns: 32-bit and 64-bit integers |
| `monotonically_increasing_id` | A unique 64-bit number per row, increasing within each partition |
| `uuid` | A random UUID string per row |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#generate-ids-and-hashes).

### `md5`, `sha2`

Cryptographic hashes as hex strings: MD5 (128-bit) and SHA-2 with 224, 256, 384 or 512 bits.

**Signature:** `F.md5(col)` · `F.sha2(col, 256)`
**Use it for:** Surrogate keys and change detection; `sha2` for pseudonymizing personal data.
**Watch out:** Hash `concat_ws` with a separator for several columns; plain concatenation makes `('ab','c')` equal `('a','bc')`.

In [ ]:
customers.select("customer_id", F.md5(F.concat_ws("|", "customer_id", "name")).alias("md5"),
                 F.sha2(F.lower("name"), 256).substr(1, 20).alias("sha256_prefix")).show(3, truncate=False)

### `hash`, `xxhash64`

Fast non-cryptographic hashes of one or more columns: 32-bit and 64-bit integers.

**Signature:** `F.hash(*cols)` · `F.xxhash64(*cols)`
**Use it for:** Bucketing and row fingerprints.
**Watch out:** 32-bit `hash` collides far sooner than `xxhash64` on large tables.

In [ ]:
customers.select("customer_id", F.hash("customer_id", "name").alias("hash"), F.xxhash64("customer_id", "name").alias("xxhash64")).show(3)

### `monotonically_increasing_id`

A unique 64-bit number per row, increasing within each partition.

**Signature:** `F.monotonically_increasing_id()`
**Use it for:** A temporary row id within one job.
**Watch out:** Not consecutive and not stable: the upper bits encode the partition, so values change when data is repartitioned.

In [ ]:
spark.range(4, numPartitions=2).select("id", F.monotonically_increasing_id().alias("mono_id")).show()

### `uuid`

A random UUID string per row. In PySpark 4.0 call it through SQL.

**Signature:** `F.expr("uuid()")`
**Use it for:** Event ids that only need to be unique, not reproducible.
**Watch out:** It changes every time the plan runs; never use it for a key you must rebuild on a rerun.

In [ ]:
spark.range(2).select("id", F.expr("uuid()").alias("uuid")).show(truncate=False)

## Array functions

Build, inspect and transform array columns. The higher-order functions (`transform`, `filter`, `exists`,
`forall`, `aggregate`) take a Python lambda that Spark turns into a SQL expression, so no Python runs per row.
The examples use `baskets`: parallel `items` and `qty` arrays, with one empty basket.

| function | what it does |
| --- | --- |
| `array` | Build an array from several columns or literals |
| `size`, `array_contains` | Number of elements, and whether the array holds a value |
| `array_distinct`, `sort_array` | Remove duplicate elements; sort the elements (ascending by default) |
| `array_join` | Join array elements into one string with a separator |
| `flatten` | Turn an array of arrays into one array |
| `arrays_zip` | Pair several arrays element by element into an array of structs |
| `element_at`, `try_element_at`, `get` | Read one element: `element_at` and `try_element_at` are 1-based (negative counts from the end); `get` is 0-based |
| `transform` | Apply a function to every element |
| `filter` | Keep the elements for which a condition is true |
| `exists`, `forall` | Whether any element, or every element, satisfies a condition |
| `aggregate` | Reduce an array to one value with a start value and a merge function |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#use-small-arrays-and-maps-in-column-expressions).

### `array`

Build an array from several columns or literals.

**Signature:** `F.array(*cols)`
**Use it for:** Collecting related fields, such as several phone numbers, into one column.
**Watch out:** All elements must have a compatible type.

In [ ]:
contacts.select("id", F.array("mobile", "home").alias("phones")).show()

### `size`, `array_contains`

Number of elements, and whether the array holds a value.

**Signature:** `F.size(col)` · `F.array_contains(col, value)`
**Use it for:** Filtering on list contents and length checks.
**Watch out:** `size` counts null elements too; the size of a null array is null in ANSI mode (-1 when ANSI is off).

In [ ]:
baskets.select("order_id", "items", F.size("items").alias("size"), F.array_contains("items", "milk").alias("has_milk")).show()

### `array_distinct`, `sort_array`

Remove duplicate elements; sort the elements (ascending by default).

**Signature:** `F.array_distinct(col)` · `F.sort_array(col, asc=True)`
**Use it for:** Tidy, comparable lists of tags.
**Watch out:** `sort_array` keeps duplicates; combine the two for a sorted distinct list.

In [ ]:
baskets.select("items", F.array_distinct("items").alias("distinct"), F.sort_array("items").alias("sorted"),
               F.sort_array("items", asc=False).alias("sorted_desc")).show(truncate=False)

### `array_join`

Join array elements into one string with a separator.

**Signature:** `F.array_join(col, sep, null_replacement=None)`
**Use it for:** Readable exports of list columns.
**Watch out:** Null elements are skipped unless you pass `null_replacement`.

In [ ]:
baskets.select("order_id", F.array_join("items", ", ").alias("joined")).show()

### `flatten`

Turn an array of arrays into one array.

**Signature:** `F.flatten(col)`
**Use it for:** Combining nested lists, for example after `collect_list` of arrays.
**Watch out:** Only one level is removed per call.

In [ ]:
spark.range(1).select(F.flatten(F.array(F.array(F.lit(1), F.lit(2)), F.array(F.lit(3)))).alias("flatten")).show()

### `arrays_zip`

Pair several arrays element by element into an array of structs.

**Signature:** `F.arrays_zip(*cols)`
**Use it for:** Exploding parallel arrays (items and quantities) together, without mixing them up.
**Watch out:** The shorter array is padded with nulls.

In [ ]:
baskets.select("order_id", F.arrays_zip("items", "qty").alias("lines")).show(truncate=False)

### `element_at`, `try_element_at`, `get`

Read one element: `element_at` and `try_element_at` are 1-based (negative counts from the end); `get` is 0-based.

**Signature:** `F.element_at(col, index)` · `F.try_element_at(col, F.lit(index))` · `F.get(col, index)`
**Use it for:** The first or last element of a list.
**Watch out:** In ANSI mode `element_at` raises an error past the end of an array; `try_element_at` and `get` return null.

In [ ]:
baskets.select("items", F.try_element_at("items", F.lit(1)).alias("first"), F.try_element_at("items", F.lit(-1)).alias("last"),
               F.get("items", 0).alias("get_0")).show()

### `transform`

Apply a function to every element.

**Signature:** `F.transform(col, lambda x: ...)`
**Use it for:** Cleaning every tag or converting every price.
**Watch out:** Use Spark functions inside the lambda (`F.trim(x)`), not Python string methods.

In [ ]:
baskets.select("qty", F.transform("qty", lambda q: q * 10).alias("qty_x10"), F.transform("items", lambda i: F.upper(i)).alias("upper_items")).show(truncate=False)

### `filter`

Keep the elements for which a condition is true.

**Signature:** `F.filter(col, lambda x: condition)`
**Use it for:** Dropping empty or invalid elements from a list.
**Watch out:** It shares its name with `DataFrame.filter`; here it works inside one array.

In [ ]:
baskets.select("qty", F.filter("qty", lambda q: q >= 2).alias("qty_ge_2")).show()

### `exists`, `forall`

Whether any element, or every element, satisfies a condition.

**Signature:** `F.exists(col, lambda x: ...)` · `F.forall(col, lambda x: ...)`
**Use it for:** "does this order contain a restricted item?" style checks.
**Watch out:** For an empty array `exists` is false and `forall` is true.

In [ ]:
baskets.select("items", F.exists("items", lambda i: i == "bread").alias("has_bread"), F.forall("qty", lambda q: q > 0).alias("all_positive")).show()

### `aggregate`

Reduce an array to one value with a start value and a merge function.

**Signature:** `F.aggregate(col, initial, lambda acc, x: ...)`
**Use it for:** Totals, products or custom reductions over a list.
**Watch out:** The start value sets the result type: start a decimal sum with a decimal zero.

In [ ]:
baskets.select("qty", F.aggregate("qty", F.lit(0), lambda total, q: total + q).alias("total_qty")).show()

## Struct and map functions

A **struct** groups named fields into one column; a **map** holds key-value pairs whose keys can differ per row.

| function | what it does |
| --- | --- |
| `struct` | Group several columns into one struct column |
| `create_map` | Build a map from alternating key and value columns |
| `map_keys`, `map_values`, `map_contains_key` | The keys or values of a map as arrays, and whether a key exists |

Explained in [PySpark 8](pyspark-08-nested-and-semi-structured-data.ipynb).

### `struct`

Group several columns into one struct column. Read a field back with a dot.

**Signature:** `F.struct(*cols)`
**Use it for:** Nested output for JSON, and carrying several fields through `max_by` or `collect_list` together.
**Watch out:** Field names come from the column names or aliases.

In [ ]:
profiles = customers.select("customer_id", F.struct("name", "region").alias("profile"))
profiles.select("customer_id", "profile", F.col("profile.name").alias("name")).show(3, truncate=False)

### `create_map`

Build a map from alternating key and value columns.

**Signature:** `F.create_map(key1, value1, key2, value2, ...)`
**Use it for:** Small lookup vocabularies and attribute bags.
**Watch out:** Keys must not be null, and all values share one type.

In [ ]:
status_names = F.create_map(F.lit("paid"), F.lit("Paid"), F.lit("shipped"), F.lit("On its way"))
orders.select("status", F.element_at(status_names, F.col("status")).alias("label")).distinct().orderBy("status").show()

### `map_keys`, `map_values`, `map_contains_key`

The keys or values of a map as arrays, and whether a key exists.

**Signature:** `F.map_keys(col)` · `F.map_values(col)` · `F.map_contains_key(col, key)`
**Use it for:** Inspecting attribute bags and telling a missing key from a null value.
**Watch out:** `element_at` returns null both for a missing key and for a key whose value is null.

In [ ]:
attrs = customers.select("customer_id", F.create_map(F.lit("region"), F.col("region"), F.lit("signup"), F.col("signup_date").cast("string")).alias("attrs"))
attrs.where("customer_id IN (1, 4)").select("customer_id", F.map_keys("attrs").alias("keys"), F.map_values("attrs").alias("values"),
    F.map_contains_key("attrs", F.lit("region")).alias("has_region_key")).show(truncate=False)

## Generator functions

Turn one row into many: one output row per array element or map entry.

| function | what it does |
| --- | --- |
| `explode`, `explode_outer` | One row per element |
| `posexplode` | Like `explode`, plus the element's position (0-based) |

Explained in [PySpark 8](pyspark-08-nested-and-semi-structured-data.ipynb).

### `explode`, `explode_outer`

One row per element. `explode` drops rows whose array is empty or null; `explode_outer` keeps them with a null element.

**Signature:** `F.explode(col)` · `F.explode_outer(col)`
**Use it for:** Flattening order lines, tags and event lists into rows.
**Watch out:** The row count multiplies; and `explode` silently loses parents with empty lists.

In [ ]:
baskets.select("order_id", F.explode("items").alias("item")).show()
baskets.select("order_id", F.explode_outer("items").alias("item")).where("order_id = 3").show()

### `posexplode`

Like `explode`, plus the element's position (0-based).

**Signature:** `F.posexplode(col)` (returns `pos` and `col`)
**Use it for:** Keeping the original order of list elements, such as line numbers.
**Watch out:** Rename the two outputs with `.alias("pos", "item")`.

In [ ]:
baskets.select("order_id", F.posexplode("items").alias("pos", "item")).show()

## JSON functions

Parse JSON text into typed columns and write columns back as JSON. The examples use `payloads`, with one
complete event, one without tags and one that is not JSON.

| function | what it does |
| --- | --- |
| `schema_of_json` | Infer a schema string from one JSON example |
| `from_json` | Parse JSON text into a struct (or array or map) using a schema |
| `to_json` | Write a struct, array or map column as JSON text |
| `get_json_object` | Extract one value from JSON text by a JSONPath such as `$.geo.country`, as a string |
| `json_tuple` | Extract several top-level keys at once, as strings |

Explained in [PySpark 3](pyspark-03-types-functions-and-nulls.ipynb#parse-and-build-json).

### `schema_of_json`

Infer a schema string from one JSON example.

**Signature:** `F.schema_of_json(F.lit(sample))`
**Use it for:** A quick start for writing the schema `from_json` needs.
**Watch out:** One sample can miss optional fields; review the result before using it.

In [ ]:
sample = payloads.first().body
print(spark.range(1).select(F.schema_of_json(F.lit(sample))).first()[0])

### `from_json`

Parse JSON text into a struct (or array or map) using a schema.

**Signature:** `F.from_json(col, schema)`
**Use it for:** Typing event payloads once, then querying fields with dots.
**Watch out:** Malformed text and mismatched schemas give nulls, not errors; count them after parsing.

In [ ]:
parsed = payloads.select("event_id", F.from_json("body", "device string, geo struct<country string>, tags array<string>").alias("data"))
parsed.select("event_id", "data.device", "data.geo.country", "data.tags").show()

### `to_json`

Write a struct, array or map column as JSON text.

**Signature:** `F.to_json(col)`
**Use it for:** Messages for Kafka or an API; one JSON document per row.
**Watch out:** Build the shape with `struct` first to control field names.

In [ ]:
orders.select(F.to_json(F.struct("order_id", "status", "amount")).alias("message")).show(3, truncate=False)

### `get_json_object`

Extract one value from JSON text by a JSONPath such as `$.geo.country`, as a string.

**Signature:** `F.get_json_object(col, "$.path")`
**Use it for:** Pulling one field without defining a schema.
**Watch out:** Every call re-parses the text; for several fields, parse once with `from_json`.

In [ ]:
payloads.select("event_id", F.get_json_object("body", "$.geo.country").alias("country"), F.get_json_object("body", "$.tags[0]").alias("first_tag")).show()

### `json_tuple`

Extract several top-level keys at once, as strings.

**Signature:** `F.json_tuple(col, *keys)`
**Use it for:** Flat JSON where a schema would be overkill.
**Watch out:** Only top-level keys, and only strings: nested objects come back as JSON text.

In [ ]:
payloads.select("event_id", F.json_tuple("body", "device", "geo").alias("device", "geo_text")).show(truncate=False)

## Aggregate functions

Reduce many rows to one value, in `df.agg(...)`, in `groupBy(...).agg(...)` or over a window. All of them skip
nulls except `count("*")`. The examples group `orders` by customer or summarize the whole table.

| function | what it does |
| --- | --- |
| `count`, `count_distinct` | Rows (`count("*")`), non-null values (`count(col)`), and distinct values |
| `approx_count_distinct` | An estimate of the distinct count using HyperLogLog, with a target relative error (`rsd`, default 0.05) |
| `sum`, `avg` / `mean`, `min`, `max` | The standard summaries |
| `count_if` | Count rows where a condition is true |
| `sum_distinct` | The sum of the distinct values |
| `product` | All values multiplied together |
| `try_sum`, `try_avg` | `sum` and `avg` that return null on overflow instead of failing |
| `collect_list` / `array_agg`, `collect_set` | Gather a group's values into an array: all values, or distinct values |
| `listagg` / `string_agg` | Join a group's values into one string with a delimiter |
| `first`, `last`, `any_value` | A value from the group: the first or last one Spark reads, or any one |
| `max_by`, `min_by` | The value of one column on the row where another column is largest or smallest |
| `mode` | The most frequent value |
| `median`, `percentile` | Exact median and exact percentiles (pass a list for several), interpolated between values |
| `percentile_approx` / `approx_percentile` | An approximate percentile that is always a value from the data; higher `accuracy` costs more memory |
| `histogram_numeric` | An approximate histogram with `n` bins, as an array of `{x, y}` structs (bin centre, count) |
| `stddev`, `variance` and their population versions | Spread: `stddev` / `std` / `stddev_samp` and `variance` / `var_samp` treat rows as a sample; `stddev_pop` and `var_pop` as the whole population |
| `skewness`, `kurtosis` | Shape: skewness measures asymmetry (positive means a long right tail); kurtosis measures how extreme the tails are |
| `corr`, `covar_samp`, `covar_pop` | Pearson correlation (-1 to 1) and covariance between two columns |
| `regr_slope`, `regr_intercept`, `regr_r2`, `regr_count`, `regr_avgx`, `regr_avgy` | A least-squares line `y = slope * x + intercept` per group, its fit (r2) and the pairs it used |
| `bool_and` / `every`, `bool_or` / `some` | Whether a condition holds for every row, or for at least one row, of the group |
| `bit_or`, `bit_and`, `bit_xor` | Combine integer flags bit by bit across a group |
| `grouping`, `grouping_id` | In `rollup`, `cube` and grouping sets, mark which rows are subtotals |

Explained in [PySpark 7](pyspark-07-aggregations-and-windows.ipynb#aggregate-function-catalog).

### `count`, `count_distinct`

Rows (`count("*")`), non-null values (`count(col)`), and distinct values.

**Signature:** `F.count("*")` · `F.count(col)` · `F.count_distinct(*cols)`
**Use it for:** Volumes, completeness and cardinality.
**Watch out:** `count(col)` and `count_distinct` skip nulls.

In [ ]:
customers.agg(F.count("*").alias("rows"), F.count("region").alias("with_region"), F.count_distinct("region").alias("regions")).show()

### `approx_count_distinct`

An estimate of the distinct count using HyperLogLog, with a target relative error (`rsd`, default 0.05).

**Signature:** `F.approx_count_distinct(col, rsd=0.05)`
**Use it for:** Distinct users per day on very large data.
**Watch out:** Estimates cannot be added together: daily estimates do not sum to a monthly one.

In [ ]:
spark.range(50_000).select((F.col("id") % 4999).alias("user")).agg(F.count_distinct("user").alias("exact"),
    F.approx_count_distinct("user").alias("approx"), F.approx_count_distinct("user", 0.01).alias("approx_rsd_1pct")).show()

### `sum`, `avg` / `mean`, `min`, `max`

The standard summaries. `mean` is an alias of `avg`.

**Signature:** `F.sum(col)` · `F.avg(col)` · `F.mean(col)` · `F.min(col)` · `F.max(col)`
**Use it for:** Every metric table.
**Watch out:** `sum` of no values is null, not 0; wrap it in `coalesce` for reports.

In [ ]:
orders.groupBy("customer_id").agg(F.sum("amount").alias("sum"), F.round(F.avg("amount"), 2).alias("avg"),
    F.min("amount").alias("min"), F.max("amount").alias("max")).orderBy("customer_id").show(3)

### `count_if`

Count rows where a condition is true.

**Signature:** `F.count_if(condition)`
**Use it for:** Several filtered counts in one pass.
**Watch out:** A null condition is not counted.

In [ ]:
orders.groupBy("customer_id").agg(F.count_if(F.col("status") == "paid").alias("paid"), F.count_if(F.col("amount") > 100).alias("over_100")).orderBy("customer_id").show(3)

### `sum_distinct`

The sum of the distinct values.

**Signature:** `F.sum_distinct(col)`
**Use it for:** Rare: only when the values themselves should be unique.
**Watch out:** It removes repeated **values**, not repeated rows; to undo duplicate rows, deduplicate by key and `sum`.

In [ ]:
spark.createDataFrame([(1, 50), (1, 50), (2, 30), (3, 30)], "payment_id int, amount int").agg(
    F.sum("amount").alias("sum"), F.sum_distinct("amount").alias("sum_distinct")).show()

### `product`

All values multiplied together.

**Signature:** `F.product(col)`
**Use it for:** Compounding growth factors.
**Watch out:** The result is a `double`.

In [ ]:
spark.createDataFrame([(1.10,), (0.95,), (1.20,)], "factor double").agg(F.round(F.product("factor"), 4).alias("product")).show()

### `try_sum`, `try_avg`

`sum` and `avg` that return null on overflow instead of failing.

**Signature:** `F.try_sum(col)` · `F.try_avg(col)`
**Use it for:** Very large integer sums where overflow is possible.
**Watch out:** A null result hides how large the true value was; cast to `decimal` or `bigint` when you need the number.

In [ ]:
spark.createDataFrame([(9223372036854775807,), (1,)], "v long").agg(F.try_sum("v").alias("try_sum"), F.try_avg("v").alias("try_avg")).show()

### `collect_list` / `array_agg`, `collect_set`

Gather a group's values into an array: all values, or distinct values.

**Signature:** `F.collect_list(col)` · `F.array_agg(col)` · `F.collect_set(col)`
**Use it for:** Lists of statuses, ids or tags per entity.
**Watch out:** Element order is not guaranteed; wrap in `array_sort`. Large groups build large arrays in one task.

In [ ]:
orders.groupBy("customer_id").agg(F.array_sort(F.collect_list("status")).alias("statuses"),
    F.array_sort(F.collect_set("status")).alias("distinct_statuses")).orderBy("customer_id").show(3, truncate=False)

### `listagg` / `string_agg`

Join a group's values into one string with a delimiter.

**Signature:** `F.listagg(col, delimiter)` · `F.string_agg(col, delimiter)`
**Use it for:** Readable report columns.
**Watch out:** Order is not guaranteed; in SQL use `listagg(col, ',') WITHIN GROUP (ORDER BY ...)`.

In [ ]:
orders.groupBy("customer_id").agg(F.listagg("status", F.lit(", ")).alias("listagg")).orderBy("customer_id").show(3, truncate=False)
spark.sql("SELECT customer_id, listagg(status, ', ') WITHIN GROUP (ORDER BY order_ts) AS ordered FROM {o} GROUP BY customer_id ORDER BY customer_id LIMIT 3", o=orders).show(truncate=False)

### `first`, `last`, `any_value`

A value from the group: the first or last one Spark reads, or any one.

**Signature:** `F.first(col, ignorenulls=False)` · `F.last(col, ignorenulls=False)` · `F.any_value(col)`
**Use it for:** Carrying an attribute that is the same on every row of the group, such as a name per id.
**Watch out:** Which row they read is not defined; to pick a specific row use `max_by`, `min_by` or a window.

In [ ]:
orders.groupBy("customer_id").agg(F.first("status").alias("first"), F.last("status").alias("last"), F.any_value("customer_id").alias("any_value")).orderBy("customer_id").show(3)

### `max_by`, `min_by`

The value of one column on the row where another column is largest or smallest.

**Signature:** `F.max_by(value_col, order_col)` · `F.min_by(value_col, order_col)`
**Use it for:** The latest status, the biggest order, the first product bought, without a join or a window.
**Watch out:** Ties pick an arbitrary row; pass a struct as `order_col` to add a tie-breaker.

In [ ]:
orders.groupBy("customer_id").agg(F.max_by("order_id", "amount").alias("biggest_order"), F.max_by("status", "order_ts").alias("latest_status"),
    F.min_by("order_id", "order_ts").alias("first_order")).orderBy("customer_id").show(3)

### `mode`

The most frequent value. `deterministic=True` breaks ties by the smallest value.

**Signature:** `F.mode(col, deterministic=False)`
**Use it for:** The usual category, store or device per customer.
**Watch out:** Without `deterministic=True`, a tie can give a different answer on each run.

In [ ]:
orders.agg(F.mode("status", deterministic=True).alias("most_common_status")).show()

### `median`, `percentile`

Exact median and exact percentiles (pass a list for several), interpolated between values.

**Signature:** `F.median(col)` · `F.percentile(col, p_or_list)`
**Use it for:** Typical values that a few extremes would distort.
**Watch out:** Exact percentiles hold all of a group's values in memory; use `percentile_approx` on big groups.

In [ ]:
orders.agg(F.median("amount").alias("median"), F.percentile("amount", [0.25, 0.75]).alias("quartiles")).show(truncate=False)

### `percentile_approx` / `approx_percentile`

An approximate percentile that is always a value from the data; higher `accuracy` costs more memory.

**Signature:** `F.percentile_approx(col, p_or_list, accuracy=10000)`
**Use it for:** P95 latency and other percentiles over billions of rows.
**Watch out:** Results can differ slightly from the exact percentile.

In [ ]:
orders.agg(F.percentile_approx("amount", 0.5).alias("approx_median"), F.approx_percentile("amount", [0.9, 0.99]).alias("p90_p99")).show(truncate=False)

### `histogram_numeric`

An approximate histogram with `n` bins, as an array of `{x, y}` structs (bin centre, count).

**Signature:** `F.histogram_numeric(col, F.lit(n))`
**Use it for:** A quick look at a distribution without leaving Spark.
**Watch out:** Bin centres are computed from the data, not fixed boundaries.

In [ ]:
orders.agg(F.histogram_numeric(F.col("amount").cast("double"), F.lit(3)).alias("histogram")).show(truncate=False)

### `stddev`, `variance` and their population versions

Spread: `stddev` / `std` / `stddev_samp` and `variance` / `var_samp` treat rows as a sample; `stddev_pop` and `var_pop` as the whole population.

**Signature:** `F.stddev(col)` · `F.stddev_pop(col)` · `F.variance(col)` · `F.var_pop(col)`
**Use it for:** Variability and outlier thresholds.
**Watch out:** A single-row group has a null sample standard deviation.

In [ ]:
orders.agg(F.round(F.stddev("amount"), 1).alias("stddev"), F.round(F.stddev_pop("amount"), 1).alias("stddev_pop"),
           F.round(F.variance("amount"), 0).alias("variance"), F.round(F.var_pop("amount"), 0).alias("var_pop")).show()

### `skewness`, `kurtosis`

Shape: skewness measures asymmetry (positive means a long right tail); kurtosis measures how extreme the tails are.

**Signature:** `F.skewness(col)` · `F.kurtosis(col)`
**Use it for:** Deciding whether an average is a fair summary.
**Watch out:** Both are sensitive to a single outlier, which is what they are meant to reveal.

In [ ]:
orders.agg(F.round(F.skewness("amount"), 2).alias("skewness"), F.round(F.kurtosis("amount"), 2).alias("kurtosis")).show()

### `corr`, `covar_samp`, `covar_pop`

Pearson correlation (-1 to 1) and covariance between two columns.

**Signature:** `F.corr(a, b)` · `F.covar_samp(a, b)` · `F.covar_pop(a, b)`
**Use it for:** Checking whether two metrics move together.
**Watch out:** Correlation does not show cause, and it only measures straight-line relationships.

In [ ]:
ads.groupBy("region").agg(F.round(F.corr("spend", "sales"), 3).alias("corr"), F.round(F.covar_samp("spend", "sales"), 1).alias("covar_samp"),
                          F.round(F.covar_pop("spend", "sales"), 1).alias("covar_pop")).orderBy("region").show()

### `regr_slope`, `regr_intercept`, `regr_r2`, `regr_count`, `regr_avgx`, `regr_avgy`

A least-squares line `y = slope * x + intercept` per group, its fit (r2) and the pairs it used.

**Signature:** `F.regr_slope(y, x)` and the others (note: **y first**)
**Use it for:** Simple trend lines per region or product without leaving Spark.
**Watch out:** Rows where either value is null are skipped by every `regr_` function.

In [ ]:
ads.groupBy("region").agg(F.round(F.regr_slope("sales", "spend"), 2).alias("slope"), F.round(F.regr_intercept("sales", "spend"), 1).alias("intercept"),
    F.round(F.regr_r2("sales", "spend"), 3).alias("r2"), F.regr_count("sales", "spend").alias("pairs"),
    F.regr_avgx("sales", "spend").alias("avg_x"), F.regr_avgy("sales", "spend").alias("avg_y")).orderBy("region").show()

### `bool_and` / `every`, `bool_or` / `some`

Whether a condition holds for every row, or for at least one row, of the group.

**Signature:** `F.bool_and(cond)` · `F.every(cond)` · `F.bool_or(cond)` · `F.some(cond)`
**Use it for:** "all orders settled?", "any return?" flags per customer.
**Watch out:** An empty or all-null group gives null.

In [ ]:
orders.groupBy("customer_id").agg(F.bool_and(F.col("status").isin("paid", "shipped")).alias("all_settled"),
    F.bool_or(F.col("status") == "returned").alias("any_return")).orderBy("customer_id").show(3)

### `bit_or`, `bit_and`, `bit_xor`

Combine integer flags bit by bit across a group.

**Signature:** `F.bit_or(col)` · `F.bit_and(col)` · `F.bit_xor(col)`
**Use it for:** Permission masks and feature flags stored as integers.
**Watch out:** They work on integer types only.

In [ ]:
grants = spark.createDataFrame([("ana", 1), ("ana", 2), ("ben", 1), ("ben", 5)], "user string, permission int")
grants.groupBy("user").agg(F.bit_or("permission").alias("bit_or"), F.bit_and("permission").alias("bit_and"), F.bit_xor("permission").alias("bit_xor")).orderBy("user").show()

### `grouping`, `grouping_id`

In `rollup`, `cube` and grouping sets, mark which rows are subtotals.

**Signature:** `F.grouping(col)` · `F.grouping_id(*cols)`
**Use it for:** Telling a subtotal row from a row whose key really is null.
**Watch out:** Never detect subtotals by testing for null keys.

In [ ]:
orders.rollup("status").agg(F.count("*").alias("orders"), F.grouping("status").alias("is_total")).orderBy("is_total", "status").show()

## Window functions

Compute a value for every row from related rows, without collapsing them. Attach each function to a window
specification with `.over(...)`. The examples use `by_customer = Window.partitionBy("customer_id").orderBy("order_ts", "order_id")`.

| function | what it does |
| --- | --- |
| `row_number` | 1, 2, 3,  |
| `rank`, `dense_rank` | Positions where ties share a rank: `rank` skips numbers after a tie (1, 1, 3), `dense_rank` does not (1, 1, 2) |
| `percent_rank`, `cume_dist` | Relative position: `(rank - 1) / (rows - 1)`, and the fraction of rows at or before this one |
| `ntile` | Split the ordered rows of each partition into `n` buckets as evenly as possible |
| `lag`, `lead` | The value `offset` rows before or after the current row, or `default` at the partition edge |
| `nth_value` | The value from the nth row of the frame (1-based), or null if the frame is shorter |
| `first`, `last` over a window | The first or last value in the frame; `ignorenulls=True` skips nulls, which gives forward fill |
| Aggregates over a window | Any aggregate with `.over(spec)`: a group value on every row, or a running value with a frame |

Explained in [PySpark window functions](pyspark-window-functions.ipynb).

### `row_number`

1, 2, 3, ... in window order, with no ties.

**Signature:** `F.row_number().over(spec)`
**Use it for:** Deduplication (keep `rn = 1`) and exactly-N-per-group results.
**Watch out:** Tied rows are numbered in an arbitrary order; add a tie-breaker to `orderBy`.

In [ ]:
orders.select("customer_id", "order_id", "order_ts", F.row_number().over(by_customer).alias("row_number")).where("customer_id <= 2").show()

### `rank`, `dense_rank`

Positions where ties share a rank: `rank` skips numbers after a tie (1, 1, 3), `dense_rank` does not (1, 1, 2).

**Signature:** `F.rank().over(spec)` · `F.dense_rank().over(spec)`
**Use it for:** Leaderboards and top-N where ties should all be kept.
**Watch out:** Pick `dense_rank` for "top N values" and `rank` for "top N places".

In [ ]:
scores = spark.createDataFrame([("Ana", 300), ("Ben", 300), ("Cai", 200), ("Dee", 100)], "rep string, revenue int")
by_revenue = Window.orderBy(F.desc("revenue"))
scores.select("rep", "revenue", F.rank().over(by_revenue).alias("rank"), F.dense_rank().over(by_revenue).alias("dense_rank")).show()

### `percent_rank`, `cume_dist`

Relative position: `(rank - 1) / (rows - 1)`, and the fraction of rows at or before this one.

**Signature:** `F.percent_rank().over(spec)` · `F.cume_dist().over(spec)`
**Use it for:** Percentile positions and top-10-percent filters.
**Watch out:** Tied rows share the same value.

In [ ]:
scores.select("rep", "revenue", F.round(F.percent_rank().over(by_revenue), 2).alias("percent_rank"),
              F.round(F.cume_dist().over(by_revenue), 2).alias("cume_dist")).show()

### `ntile`

Split the ordered rows of each partition into `n` buckets as evenly as possible.

**Signature:** `F.ntile(n).over(spec)`
**Use it for:** Quartiles and deciles of customers by spend.
**Watch out:** When rows do not divide evenly, the first buckets get the extra rows.

In [ ]:
orders.select("order_id", "amount", F.ntile(4).over(Window.orderBy("amount")).alias("quartile")).orderBy("amount").show(6)

### `lag`, `lead`

The value `offset` rows before or after the current row, or `default` at the partition edge.

**Signature:** `F.lag(col, offset=1, default=None).over(spec)` · `F.lead(col, offset=1, default=None).over(spec)`
**Use it for:** Time since the previous event, change from the previous value, the next status.
**Watch out:** Without `partitionBy`, `lag` reads across entities at the boundaries.

In [ ]:
orders.select("customer_id", "order_id", "amount", F.lag("amount").over(by_customer).alias("prev_amount"),
              F.lead("order_id", 1, -1).over(by_customer).alias("next_order")).where("customer_id <= 2").show()

### `nth_value`

The value from the nth row of the frame (1-based), or null if the frame is shorter.

**Signature:** `F.nth_value(col, n, ignoreNulls=False).over(spec)`
**Use it for:** Each customer's second order, a common retention metric.
**Watch out:** With the default frame, rows before the nth get null; use a whole-partition frame to fill every row.

In [ ]:
whole = by_customer.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
orders.select("customer_id", "order_id", F.nth_value("order_id", 2).over(by_customer).alias("default_frame"),
              F.nth_value("order_id", 2).over(whole).alias("whole_partition")).where("customer_id <= 2").show()

### `first`, `last` over a window

The first or last value in the frame; `ignorenulls=True` skips nulls, which gives forward fill.

**Signature:** `F.first(col, ignorenulls).over(spec)` · `F.last(col, ignorenulls).over(spec)`
**Use it for:** First and latest values per entity, and filling gaps in sparse readings.
**Watch out:** With `orderBy` and no frame, `last` returns the current row; set `rowsBetween` explicitly.

In [ ]:
orders.select("customer_id", "order_id", F.first("order_id").over(by_customer).alias("first"), F.last("order_id").over(by_customer).alias("last_default"),
              F.last("order_id").over(whole).alias("last_whole")).where("customer_id <= 2").show()

### Aggregates over a window

Any aggregate with `.over(spec)`: a group value on every row, or a running value with a frame.

**Signature:** `F.sum(col).over(spec)`, `F.avg(col).over(spec.rowsBetween(-2, 0))`, ...
**Use it for:** Shares of a total, running totals and moving averages.
**Watch out:** Adding `orderBy` changes the default frame, turning a group total into a running total.

In [ ]:
orders.select("customer_id", "order_id", "amount", F.sum("amount").over(Window.partitionBy("customer_id")).alias("customer_total"),
              F.sum("amount").over(by_customer.rowsBetween(Window.unboundedPreceding, 0)).alias("running_total")).where("customer_id <= 2").show()

## Misc and UDF functions

File lineage, partition ids, and Python functions run inside Spark.

| function | what it does |
| --- | --- |
| `input_file_name` | The path of the file each row was read from (empty for data that did not come from files) |
| `spark_partition_id` | The id of the partition each row is in |
| `udf` | Wrap a Python function so Spark calls it once per row |
| `pandas_udf` | A vectorized UDF: the Python function receives and returns whole pandas Series, moved through Arrow |

Explained in [PySpark 9](pyspark-09-python-udfs-and-pandas.ipynb).

### `input_file_name`

The path of the file each row was read from (empty for data that did not come from files).

**Signature:** `F.input_file_name()`
**Use it for:** Lineage columns in bronze tables, to trace a bad row back to its source file.
**Watch out:** File sources also expose `_metadata.file_path`, which carries more detail.

In [ ]:
drop_dir = os.path.join(WORK_DIR, "drops")
orders.select("order_id", "amount").repartition(2).write.mode("overwrite").csv(drop_dir, header=True)
landed = spark.read.csv(drop_dir, header=True, schema="order_id int, amount decimal(12,2)")
landed.select("order_id", F.substring(F.element_at(F.split(F.input_file_name(), "/"), -1), 1, 10).alias("file")).orderBy("order_id").show(3)

### `spark_partition_id`

The id of the partition each row is in.

**Signature:** `F.spark_partition_id()`
**Use it for:** Checking how evenly data is spread before a heavy step.
**Watch out:** Ids change whenever the data is repartitioned or shuffled.

In [ ]:
orders.repartition(3, "customer_id").groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

### `udf`

Wrap a Python function so Spark calls it once per row.

**Signature:** `F.udf(function, returnType)` or the `@F.udf("type")` decorator
**Use it for:** Logic no built-in function can express, such as a specialist library call.
**Watch out:** Spark cannot optimize through a UDF and must move every row to Python; check for a built-in first.

In [ ]:
@F.udf("string")
def mask_name(name):
    return None if name is None else name[0] + "*" * (len(name) - 1)

customers.select("name", mask_name("name").alias("masked")).show(3)

### `pandas_udf`

A vectorized UDF: the Python function receives and returns whole pandas Series, moved through Arrow.

**Signature:** `@F.pandas_udf("type")` on a function `pd.Series -> pd.Series`
**Use it for:** Numeric Python logic on many rows, usually much faster than `udf`.
**Watch out:** Needs `pandas` and `pyarrow`; decimals arrive as Python `Decimal` objects, so convert them deliberately.

In [ ]:
import pandas as pd

@F.pandas_udf("double")
def with_vat(amount: pd.Series) -> pd.Series:
    return amount.astype(float) * 1.2

orders.select("order_id", "amount", with_vat("amount").alias("gross")).show(3)

**Next:** see these functions in context in [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb), [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb) and [PySpark window functions](pyspark-window-functions.ipynb), or go back to the one-page [PySpark reference](pyspark-reference.ipynb).